# Flight Delay Data Merging

This notebook performs Steps 8-12 of the pipeline:
- Merge cleaned flights with airline names
- Merge origin airport location metadata
- Select final analysis columns
- Save `merged_data.csv` and verify outputs

Run `data_cleaning.ipynb` first to generate `cleaned_data.csv`.

## Step 0 - Imports and Configuration

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.options.mode.copy_on_write = True

ROOT = Path.cwd()
if not (ROOT / "data/raw/airports.csv").exists() and (ROOT.parent / "data/raw/airports.csv").exists():
    ROOT = ROOT.parent

cleaned_path = ROOT / "data/processed/cleaned_data.csv"
airports_path = ROOT / "data/raw/airports.csv"
airlines_path = ROOT / "data/raw/airlines.csv"
merged_output = ROOT / "data/processed/merged_data.csv"

print("Using root:", ROOT)

Using root: /Users/adityasingh/DV


/var/folders/mb/xcn1fzjx76ndcb617gpnmf3c0000gn/T/ipykernel_95765/1200179809.py:6: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  pd.options.mode.copy_on_write = True


## Step 1 - Load Datasets

In [ ]:
cleaned = pd.read_csv(cleaned_path, low_memory=False)
airports = pd.read_csv(airports_path, low_memory=False)
airlines = pd.read_csv(airlines_path, low_memory=False)

print("Cleaned flights:", cleaned.shape)
print("Airports:", airports.shape)
print("Airlines:", airlines.shape)

## Step 2 - Merge Airline Information

In [ ]:
cleaned["AIRLINE"] = cleaned["AIRLINE"].astype(str).str.strip().str.upper()

airlines_lookup = airlines.rename(columns={
    "IATA_CODE": "AIRLINE",
    "AIRLINE": "AIRLINE_NAME",
})
airlines_lookup["AIRLINE"] = airlines_lookup["AIRLINE"].astype(str).str.strip().str.upper()

merged = cleaned.merge(airlines_lookup, on="AIRLINE", how="left")
print("After airline merge:", merged.shape)

After airline merge: (5574612, 35)


## Step 3 - Merge Origin Airport Information

In [ ]:
merged["ORIGIN_AIRPORT"] = merged["ORIGIN_AIRPORT"].astype(str).str.strip().str.upper()

origin_lookup = airports.rename(columns={
    "IATA_CODE": "ORIGIN_AIRPORT",
    "CITY": "ORIGIN_CITY",
    "STATE": "ORIGIN_STATE",
    "LATITUDE": "ORIGIN_LATITUDE",
    "LONGITUDE": "ORIGIN_LONGITUDE",
})
origin_lookup["ORIGIN_AIRPORT"] = origin_lookup["ORIGIN_AIRPORT"].astype(str).str.strip().str.upper()

merged = merged.merge(origin_lookup, on="ORIGIN_AIRPORT", how="left")
print("After airport merge:", merged.shape)

After airport merge: (5574612, 47)


## Step 4 - Select Final Analysis Columns

In [ ]:
final_columns = [
    "YEAR",
    "MONTH",
    "DAY_OF_WEEK",
    "AIRLINE",
    "AIRLINE_NAME",
    "FLIGHT_NUMBER",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ORIGIN_CITY",
    "ORIGIN_STATE",
    "scheduled_dep_hour",
    "DEPARTURE_DELAY",
    "ARRIVAL_DELAY",
    "DISTANCE",
    "DIVERTED",
    "CANCELLED",
    "CANCELLATION_REASON",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
    "route",
    "delay_severity"
]

# Keep only columns that exist in merged dataframe
final_columns = [c for c in final_columns if c in merged.columns]

# Create final dataframe
final_df = merged[final_columns].copy()

print("Final shape:", final_df.shape)

Final shape: (5574612, 22)


## Step 5 - Save Merged Dataset and Final Verification

In [ ]:
def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)

final_df.to_csv(merged_output, index=False)

print("Saved:", merged_output)
print("Merged shape:", final_df.shape)
print(f"Merged memory usage: {memory_usage_mb(final_df):.2f} MB")
display(final_df.head())
display(final_df.dtypes)

Saved: /Users/adityasingh/DV/data/processed/merged_data.csv
Merged shape: (5574612, 22)
Merged memory usage: 2469.87 MB


,YEAR,MONTH,DAY_OF_WEEK,AIRLINE,AIRLINE_NAME,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,scheduled_dep_hour,DEPARTURE_DELAY,...,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,route,delay_severity
0,2015,1,4,AS,Alaska Airlines Inc.,98,ANC,SEA,0,-11.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,ANC-SEA,Minor
1,2015,1,4,AA,American Airlines Inc.,2336,LAX,PBI,0,-8.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,LAX-PBI,Minor
2,2015,1,4,US,US Airways Inc.,840,SFO,CLT,0,-2.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,SFO-CLT,Minor
3,2015,1,4,AA,American Airlines Inc.,258,LAX,MIA,0,-5.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,LAX-MIA,Minor
4,2015,1,4,AS,Alaska Airlines Inc.,135,SEA,ANC,0,-1.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,SEA-ANC,Minor


YEAR                     int64
MONTH                    int64
DAY_OF_WEEK              int64
AIRLINE                    str
AIRLINE_NAME               str
FLIGHT_NUMBER            int64
ORIGIN_AIRPORT             str
DESTINATION_AIRPORT        str
scheduled_dep_hour       int64
DEPARTURE_DELAY        float64
ARRIVAL_DELAY          float64
DISTANCE               float64
DIVERTED                 int64
CANCELLED                int64
CANCELLATION_REASON    float64
AIR_SYSTEM_DELAY       float64
SECURITY_DELAY         float64
AIRLINE_DELAY          float64
LATE_AIRCRAFT_DELAY    float64
WEATHER_DELAY          float64
route                      str
delay_severity             str
dtype: object